# 07 · Stratification

**Question.** How do the runs perform inside and outside the GIAB stratification regions (difficult regions, low mappability plus segmental duplications, segmental duplications)?

**Reviewer comment.** None.

**Two BED filters** (docs/legacy-deviations.md D1).
- `io.bed_filter` matches a variant only against regions on its own chromosome. It is the analysis filter: rows with `filter == "corrected"`.
- `io.bed_filter_legacy` reproduces the legacy filter, which compared positions across chromosomes. It is used to check parity with the published `union_metrics.csv` and to fill the comparison table.

**Result of D1.** The corrected filter changes the region-stratified results substantially. Published region-stratified values require revision. `tables/legacy_vs_corrected.csv` gives the per-region counts under each filter.

**Fixed as in legacy, not decided here.**
- Region boundary `start <= POS < end` (D2).
- Every metric is computed against the whole truth set, not the truth inside each region (D9).
- Precision and F1 are undefined for a run with no calls in a region. Legacy raises `ZeroDivisionError`; here they are left blank and counted in the audit.

**Reads.** The variant-set cache from notebook 01; no run VCF is parsed. The truth VCF is parsed here because no truth-set cache exists yet (it needs raw VCF fields).

**Does not.** Cover indels. The unfiltered whole-study rows of `union_metrics.csv` are used only as a parity check on the truth set and the metrics.

## Imports

In [ ]:
import sys
from pathlib import Path

try:
    import swb  # noqa: F401
except ImportError:  # swb is not pip-installed: use the repository's src/ (notebooks run from notebooks/)
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from swb import audit, config, io, metrics, viz

## Config

In [ ]:
OUT = config.results_dir("07_stratification")
STAGE = audit.Stage(OUT / ".staging")

EXPECTED_RUNS = 480
REGIONS = sorted([
    "GRCh38_alldifficultregions", "GRCh38_notinalldifficultregions",
    "GRCh38_alllowmapandsegdupregions", "GRCh38_notinalllowmapandsegdupregions",
    "GRCh38_segdups", "GRCh38_notinsegdups",
])
FILTERS = {
    "corrected": lambda vcf, regions: io.bed_filter(vcf, regions),
    "legacy": lambda vcf, regions: io.bed_filter_legacy(vcf, regions, False),
}
TOLERANCE = 1e-12  # recomputed metrics differ from the legacy CSV by about 1e-16
print("SWB_DATA_ROOT:", config.DATA_ROOT)

## Load + validate

In [ ]:
assert config.SETS_PARQUET.exists(), "run 01_variant_sets first"
sets = io.load_sets(config.SETS_PARQUET)
manifest = pd.read_csv(config.MANIFEST, dtype={"TestCaseNo": str})
assert len(sets) == EXPECTED_RUNS and set(sets) == set(manifest["TestCaseNo"])
assert all(sets.values()), "a run has no variants"

bed_paths = io.bed_files(str(config.STRATIFICATION_DIR))
assert sorted(os.path.basename(p)[:-len(".bed")] for p in bed_paths) == REGIONS, bed_paths
beds = {}
for region in REGIONS:
    bed = io.read_bed(str(config.STRATIFICATION_DIR / (region + ".bed")))
    assert bed[["start", "end"]].notna().all().all() and (bed["start"] < bed["end"]).all(), region
    beds[region] = io.BedRegions(bed)
    print("{:40} {:>9} regions".format(region, len(bed)))

truth = io.truth_set(str(config.TRUTH_VCF))
assert len(truth) > 0
print(len(sets), "runs,", sum(len(s) for s in sets.values()), "variants,", len(truth), "truth variants")

## Analysis

In [ ]:
def scores(pred):
    """(IoU, precision, recall, F1) against the whole truth set, legacy definitions.

    Precision and F1 are undefined for an empty prediction (legacy raises ZeroDivisionError): NaN.
    """
    if not pred:
        return metrics.jaccard(pred, truth), float("nan"), metrics.recall(pred, truth), float("nan")
    p, r, f = metrics.prf1(pred, truth)
    return metrics.jaccard(pred, truth), p, r, f


rows = []
for key in tqdm(sorted(sets, key=int)):
    vcf = io.variants_to_df(sets[key])
    for region in REGIONS:
        for name, keep in FILTERS.items():
            kept = io.df_to_variants(keep(vcf, beds[region]))
            rows.append((int(key), key, region, name, len(sets[key]), len(kept)) + scores(kept))
by_run = pd.DataFrame(rows, columns=["run", "TestCaseNo", "region", "filter", "n_input", "n_kept",
                                     "iou", "precision", "recall", "f1"])
assert len(by_run) == EXPECTED_RUNS * len(REGIONS) * len(FILTERS)
assert (by_run.loc[by_run["filter"] == "legacy", "n_kept"] > 0).all(), "legacy would have raised ZeroDivisionError"

comparison = []
for region in REGIONS:
    legacy = by_run[(by_run["region"] == region) & (by_run["filter"] == "legacy")]
    corrected = by_run[(by_run["region"] == region) & (by_run["filter"] == "corrected")]
    assert (legacy["n_input"].values == corrected["n_input"].values).all()
    n_in, n_leg, n_cor = int(legacy["n_input"].sum()), int(legacy["n_kept"].sum()), int(corrected["n_kept"].sum())
    comparison.append((region, len(legacy), n_in, n_leg, n_cor, n_leg - n_cor,
                       n_leg / n_in, n_cor / n_in, int((corrected["n_kept"] == 0).sum())))
comparison = pd.DataFrame(comparison, columns=[
    "region", "runs", "variants_in", "kept_legacy", "kept_corrected", "kept_difference",
    "fraction_legacy", "fraction_corrected", "runs_empty_corrected"])
comparison

## Save tables

Written to staging; moved to `results/07_stratification/tables/` only if parity passes.

In [ ]:
io.write_csv(comparison, STAGE.path(OUT / "tables" / "legacy_vs_corrected.csv"), sort_by="region")
io.write_csv(by_run, STAGE.path(OUT / "tables" / "stratified_metrics.csv"), sort_by=["run", "region", "filter"])

## Figures

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(REGIONS))
ax.barh(y - 0.2, comparison["fraction_legacy"], 0.4, label="legacy (chromosome-blind)")
ax.barh(y + 0.2, comparison["fraction_corrected"], 0.4, label="corrected (chromosome-aware)")
ax.set_yticks(y)
ax.set_yticklabels([viz.clean_bed_name(r).replace("\n", " ") for r in comparison["region"]], fontsize=7)
ax.set_xlabel("Fraction of variants kept ({} runs pooled)".format(EXPECTED_RUNS))
ax.set_title("Variants kept per region")
ax.legend(fontsize=7)
viz.save_fig(fig, STAGE.stem(OUT / "figures" / "kept_fraction_legacy_vs_corrected"))
plt.show()

## Parity check

The legacy filter must reproduce the published `union_metrics.csv`: all 6 regions for every run, plus the unfiltered whole-study rows (a check on the truth set and the metrics). The corrected filter is not compared with legacy here; that is `legacy_vs_corrected.csv`.

In [ ]:
legacy = pd.read_csv(config.LEGACY_UNION_METRICS, dtype={"TestCase": str})
legacy["region"] = legacy["Region"].str.replace(".bed", "", regex=False)
metric_cols = {"iou": "Inter/Union", "precision": "Precision", "recall": "Recall", "f1": "f1_score"}

parity = audit.Parity()

# region rows, computed with the legacy filter
published = legacy[legacy["region"].isin(REGIONS)]
parity.check("legacy has one row per run and region",
             len(published) == EXPECTED_RUNS * len(REGIONS) and not published.duplicated(["TestCase", "region"]).any(),
             "{} rows".format(len(published)))
ours = by_run[by_run["filter"] == "legacy"].merge(
    published, left_on=["TestCaseNo", "region"], right_on=["TestCase", "region"], how="outer", indicator=True)
parity.check("legacy filter covers exactly the published (run, region) rows", (ours["_merge"] == "both").all(),
             ours["_merge"].value_counts().to_dict())
for mine, theirs in metric_cols.items():
    diff = (ours[mine] - ours[theirs]).abs()
    parity.check("legacy filter: {} == union_metrics.csv {}".format(mine, theirs), (diff <= TOLERANCE).all(),
                 "max |diff| {:.2e}".format(diff.max()))

# whole-study rows: the unfiltered sets against the truth set
whole = legacy[legacy["Region"].str.lower() == "findings from this study"]
unfiltered = pd.DataFrame({k: scores(s) for k, s in sets.items()}, index=list(metric_cols)).T
worst = max(float(np.abs(whole[theirs].values - unfiltered.loc[whole["TestCase"], mine].values).max())
            for mine, theirs in metric_cols.items())
parity.check("whole-study rows == unfiltered sets vs truth set (all {} rows)".format(len(whole)),
             set(whole["TestCase"]) == set(sets) and worst <= TOLERANCE, "max |diff| {:.2e}".format(worst))

parity.finish(OUT / "audit" / "parity.txt", STAGE)

## Audit summary

In [ ]:
empty = by_run[(by_run["filter"] == "corrected") & (by_run["n_kept"] == 0)]
lines = [
    "runs: {}; regions: {}; truth variants: {}".format(len(sets), len(REGIONS), len(truth)),
    "filters: corrected = io.bed_filter (analysis); legacy = io.bed_filter_legacy (parity and comparison only)",
    "legacy filter matches positions across chromosomes (D1); both filters use start <= POS < end",
    "(D2, undecided); metrics use the whole truth set (D9)",
    "corrected: {} (run, region) pairs keep no variants, precision and F1 left blank: {}".format(
        len(empty), empty.groupby("region")["TestCaseNo"].apply(lambda s: sorted(s, key=int)).to_dict()),
    "",
    "variants kept, all runs pooled:",
    comparison.to_string(index=False),
    "",
    "python {} | pandas {} | numpy {}".format(sys.version.split()[0], pd.__version__, np.__version__),
]
(OUT / "audit").mkdir(parents=True, exist_ok=True)
(OUT / "audit" / "summary.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))